In [1]:
#%pip install paramiko

In [2]:
import base64
import getpass
import hashlib
import os
import posixpath
import sys
import threading
import time
from pathlib import Path

import pandas as pd
import paramiko
import urllib3
from dotenv import load_dotenv
from IPython import get_ipython
from selenium import webdriver
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    ElementNotInteractableException,
    InvalidElementStateException,
    InvalidSessionIdException,
    NoSuchElementException,
    NoSuchWindowException,
    StaleElementReferenceException,
    TimeoutException,
    WebDriverException,
)
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager


In [3]:
load_dotenv(dotenv_path=Path('.env'), override=True)

sftp_host = os.getenv('SFTP_HOST')
sftp_port = int(os.getenv('SFTP_PORT') or 22)
sftp_login = os.getenv('SFTP_LOGIN')
sftp_host_key_fingerprint = os.getenv('SFTP_SSHHOSTKEYFINGERPRINT')
sftp_key_path = os.getenv('SFTP_KEY')
sftp_key_passphrase = os.getenv('SFTP_KEY_PASSPHRASE')
sftp_password = os.getenv('SFTP_PASSWORD')
sftp_base_folder = os.getenv('SFTP_FOLDER')
tmdb_login = os.getenv('TMDB_LOGIN')
tmdb_password = os.getenv('TMDB_PASSWORD')

# SFTP_PASSWORD is optional now: the connection authenticates with an SSH key
# first (SFTP_LOGIN + SFTP_SSHHOSTKEYFINGERPRINT) and only falls back to the
# password when one is present in the .env.
if not sftp_host or not sftp_login or not sftp_base_folder:
    raise ValueError('Missing SFTP configuration in .env: SFTP_HOST, SFTP_LOGIN, SFTP_FOLDER')

if not tmdb_login or not tmdb_password:
    raise ValueError('Missing TMDB configuration in .env: TMDB_LOGIN, TMDB_PASSWORD')


In [4]:
DATASETS = [
    {
        'name': 'movies',
        'remote_folder': 'wikidata-id-movie-fix',
        'local_folder': 'wikidata-id-movie-fix',
        'local_file_basename': 'wikidata-id-movie-fix',
        'entity_path': 'movie',
        'id_column': 'ID_MOVIE',
        'erase_column': 'ID_MOVIE_ERASE_WIKIDATA_ID',
        'store_key': 'lngmovieidstart',
        'missing_entity_message': 'Colonne ID_MOVIE manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for movie {lngid}. Skipping.'
    },
    {
        'name': 'series',
        'remote_folder': 'wikidata-id-serie-fix',
        'local_folder': 'wikidata-id-serie-fix',
        'local_file_basename': 'wikidata-id-serie-fix',
        'entity_path': 'tv',
        'id_column': 'ID_SERIE',
        'erase_column': 'ID_SERIE_ERASE_WIKIDATA_ID',
        'store_key': 'lngserieidstart',
        'missing_entity_message': 'Colonne ID_SERIE manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for serie {lngid}. Skipping.'
    },
    {
        'name': 'persons',
        'remote_folder': 'wikidata-id-person-fix',
        'local_folder': 'wikidata-id-person-fix',
        'local_file_basename': 'wikidata-id-person-fix',
        'entity_path': 'person',
        'id_column': 'ID_PERSON',
        'erase_column': 'ID_PERSON_ERASE_WIKIDATA_ID',
        'store_key': 'lngpersonidstart',
        'missing_entity_message': 'Colonne ID_PERSON manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for person {lngid}. Skipping.',
        'intgoingdown': True
    },
    # ------------------------------------------------------------------
    # LES TROIS JEUX DE REMPLACEMENT, ajoutes le 2026-09-05.
    #
    # Les trois premiers REMPLISSENT une case wikidata_id vide ou illisible.
    # Ceux-ci REMPLACENT une valeur valide par une autre, ce qui detruit de
    # l'information sur un site tiers au lieu d'en ajouter. Leur source est
    # wikidata-id-<entite>-replace.sql, qui n'exporte que les cas ou Wikidata
    # designe elle-meme la fiche TMDb visee (garde 3 : la corroboration).
    #
    # Jeux SEPARES et non lignes supplementaires dans les fichiers -fix, pour
    # que l'on puisse lancer les remplissages sans les remplacements, mesurer
    # les deux populations a part, et interrompre l'un sans perdre le curseur
    # de l'autre. D'ou aussi des store_key distincts.
    #
    # Volumetrie attendue, mesuree le 2026-09-05 sur les films : 659 fiches,
    # contre 19 196 pour le remplissage. Le passage est donc court.
    # ------------------------------------------------------------------
    {
        'name': 'movies_replace',
        'remote_folder': 'wikidata-id-movie-replace',
        'local_folder': 'wikidata-id-movie-replace',
        'local_file_basename': 'wikidata-id-movie-replace',
        'entity_path': 'movie',
        'id_column': 'ID_MOVIE',
        'erase_column': 'ID_MOVIE_ERASE_WIKIDATA_ID',
        'store_key': 'lngmoviereplaceidstart',
        'missing_entity_message': 'Colonne ID_MOVIE manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for movie {lngid}. Skipping.'
    },
    {
        'name': 'series_replace',
        'remote_folder': 'wikidata-id-serie-replace',
        'local_folder': 'wikidata-id-serie-replace',
        'local_file_basename': 'wikidata-id-serie-replace',
        'entity_path': 'tv',
        'id_column': 'ID_SERIE',
        'erase_column': 'ID_SERIE_ERASE_WIKIDATA_ID',
        'store_key': 'lngseriereplaceidstart',
        'missing_entity_message': 'Colonne ID_SERIE manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for serie {lngid}. Skipping.'
    },
    {
        'name': 'persons_replace',
        'remote_folder': 'wikidata-id-person-replace',
        'local_folder': 'wikidata-id-person-replace',
        'local_file_basename': 'wikidata-id-person-replace',
        'entity_path': 'person',
        'id_column': 'ID_PERSON',
        'erase_column': 'ID_PERSON_ERASE_WIKIDATA_ID',
        'store_key': 'lngpersonreplaceidstart',
        'missing_entity_message': 'Colonne ID_PERSON manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for person {lngid}. Skipping.'
    }
]


In [5]:
SAVE_BUTTON_XPATH = '//input[@value="Save"]'

# TMDB's cookie banner is OneTrust, injected after the page loads and anchored at
# the bottom of the window, which is where the Save button of an edit page sits:
# while it is up, a click aimed at Save lands on the banner instead. 'Tout
# refuser' is #onetrust-reject-all-handler, and declining sets
# OptanonAlertBoxClosed for the profile, so it has to be done once per browser,
# which means once per restart too.
COOKIE_BANNER_ID = 'onetrust-banner-sdk'
COOKIE_DECLINE_ID = 'onetrust-reject-all-handler'
COOKIE_BANNER_TIMEOUT = 10

# Chrome dies on long runs: a crashed tab, a renderer out of memory, a driver
# that stops answering. These bounds decide how fast the loop notices, and how
# it tells a dead browser from a slow page. PAGE_LOAD_TIMEOUT stays below
# HTTP_CLIENT_TIMEOUT on purpose, so a slow page raises TimeoutException from
# chromedriver instead of a client read timeout that looks like a crash.
HTTP_CLIENT_TIMEOUT = 60
PAGE_LOAD_TIMEOUT = 45
SCRIPT_TIMEOUT = 30
BROWSER_QUIT_TIMEOUT = 15
BROWSER_RESTART_ATTEMPTS = 3
LOGIN_CONFIRM_TIMEOUT = 120

# Every TMDB page is same-site, so Chrome serves the whole run from a single
# renderer whose memory only grows: measured over 12 navigations, 16 MB per page
# as Chrome is launched by default, 9 MB with the back/forward cache off, 4 MB
# with the memory purge on top. Nothing reclaims all of it, so the browser is
# recycled every BROWSER_RECYCLE_EVERY records. Raise it if the extra logins are
# a nuisance, set it to 0 to turn recycling off.
BROWSER_RECYCLE_EVERY = 250

# A record that writes nothing is normal (locked field, 404); a run of them is
# not. Over the 1061 records of the previous run, 1046 were written and the
# longest run without a write was 1, so a streak this long means the browser or
# TMDB stopped cooperating and the loop must not keep marking records done.
MAX_CONSECUTIVE_SKIPS = 8

# chromedriver reports a lost browser in the message rather than in the exception
# type, so the wording is what gets matched.
BROWSER_GONE_MESSAGES = (
    'invalid session id',
    'no such session',
    'session deleted',
    'chrome not reachable',
    'disconnected',
    'target crashed',
    'tab crashed',
    'unable to connect to renderer',
    'not connected to devtools',
    'failed to connect',
)


class BrowserSessionLost(RuntimeError):
    """Chrome, chromedriver or the TMDB session is gone: the row needs a new browser."""


SSH_CONFIG_PATH = Path.home() / '.ssh' / 'config'


class HostKeyVerificationError(paramiko.SSHException):
    """The SFTP server host key does not match SFTP_SSHHOSTKEYFINGERPRINT."""


def parse_host_key_fingerprint(raw_fingerprint):
    """Split a WinSCP-style fingerprint into (key_type, algorithm, digest).

    The key type and the bit count are optional, so the same .env value can be
    shared with the WinSCP/PowerShell scripts. Accepted shapes:
        ssh-ed25519 256 SHA256:<base64>
        ssh-ed25519 256 <base64>
        SHA256:<base64>
        ssh-rsa 2048 aa:bb:...:ff      (MD5)
    """
    tokens = raw_fingerprint.strip().split()
    if not tokens:
        raise ValueError('SFTP_SSHHOSTKEYFINGERPRINT is empty')

    key_type = tokens[0] if len(tokens) > 1 else None
    digest = tokens[-1]

    if digest.upper().startswith('SHA256:'):
        return key_type, 'sha256', digest[len('SHA256:'):].rstrip('=')
    if digest.upper().startswith('MD5:'):
        digest = digest[len('MD5:'):]
    if ':' in digest:
        return key_type, 'md5', digest.replace(':', '').lower()
    return key_type, 'sha256', digest.rstrip('=')


def compute_host_key_digest(host_key, algorithm):
    if algorithm == 'md5':
        return hashlib.md5(host_key.asbytes()).hexdigest()
    return base64.b64encode(hashlib.sha256(host_key.asbytes()).digest()).decode('ascii').rstrip('=')


class FingerprintHostKeyPolicy(paramiko.MissingHostKeyPolicy):
    """Pin the server to SFTP_SSHHOSTKEYFINGERPRINT instead of known_hosts.

    Same contract as WinSCP's SessionOptions.SshHostKeyFingerprint.
    """

    def __init__(self, raw_fingerprint):
        self.key_type, self.algorithm, self.digest = parse_host_key_fingerprint(raw_fingerprint)

    def missing_host_key(self, client, hostname, key):
        if self.key_type and key.get_name() != self.key_type:
            raise HostKeyVerificationError(
                f'SSH host key type mismatch: the server offered {key.get_name()}, '
                f'SFTP_SSHHOSTKEYFINGERPRINT declares {self.key_type}.'
            )

        actual_digest = compute_host_key_digest(key, self.algorithm)
        if actual_digest != self.digest:
            raise HostKeyVerificationError(
                f'SSH host key fingerprint mismatch ({self.algorithm}): the server presented '
                f'{actual_digest}, SFTP_SSHHOSTKEYFINGERPRINT expects {self.digest}. '
                'Refusing to connect.'
            )


def collect_private_key_files():
    """Private keys to offer, most specific first.

    paramiko only auto-discovers the default ~/.ssh/id_rsa|id_ecdsa|id_ed25519
    names, so the ~/.ssh/config lookup is what makes a per-host key such as
    id_ed25519_<vps> usable. 'IdentitiesOnly yes' is honoured so the server is
    not offered every key on the machine (MaxAuthTries).
    """
    if sftp_key_path:
        return [str(Path(sftp_key_path).expanduser())]

    candidates = []
    identities_only = False

    if SSH_CONFIG_PATH.is_file():
        ssh_config = paramiko.SSHConfig()
        with SSH_CONFIG_PATH.open(encoding='utf-8') as config_file:
            ssh_config.parse(config_file)
        host_config = ssh_config.lookup(sftp_host)
        identities_only = str(host_config.get('identitiesonly', 'no')).lower() in ('yes', 'true')
        candidates.extend(Path(name).expanduser() for name in host_config.get('identityfile', []))

    if not (identities_only and candidates):
        ssh_dir = Path.home() / '.ssh'
        if ssh_dir.is_dir():
            candidates.extend(sorted(ssh_dir.glob('id_*')))

    key_files = []
    for candidate in candidates:
        if candidate.suffix == '.pub' or not candidate.is_file():
            continue
        if str(candidate) not in key_files:
            key_files.append(str(candidate))
    return key_files


def is_encrypted_private_key(key_file):
    for key_class in (paramiko.Ed25519Key, paramiko.RSAKey, paramiko.ECDSAKey):
        try:
            key_class.from_private_key_file(key_file)
            return False
        except paramiko.PasswordRequiredException:
            return True
        except Exception:
            continue
    return False


KEY_PASSPHRASE_CACHE = {}


def resolve_key_passphrase(key_files):
    """SFTP_KEY_PASSPHRASE when set, otherwise ask once for a protected key.

    Prompting keeps the passphrase out of the .env and out of the notebook
    output; the answer is cached so a run over the three datasets only asks
    once. Clear KEY_PASSPHRASE_CACHE to be prompted again.
    """
    if sftp_key_passphrase:
        return sftp_key_passphrase

    if 'passphrase' in KEY_PASSPHRASE_CACHE:
        return KEY_PASSPHRASE_CACHE['passphrase']

    if not any(is_encrypted_private_key(key_file) for key_file in key_files):
        return None

    if get_ipython() is None and not sys.stdin.isatty():
        return None

    KEY_PASSPHRASE_CACHE['passphrase'] = getpass.getpass(
        'Passphrase for the SSH private key (empty to skip key auth): '
    ) or None
    return KEY_PASSPHRASE_CACHE['passphrase']


def make_transport_factory():
    """Negotiate the host key type declared in SFTP_SSHHOSTKEYFINGERPRINT first.

    Without known_hosts entries paramiko would otherwise pick its own preferred
    algorithm, which may not be the one the fingerprint was taken from.
    """
    key_type = parse_host_key_fingerprint(sftp_host_key_fingerprint)[0] if sftp_host_key_fingerprint else None

    def transport_factory(sock, *args, **kwargs):
        transport = paramiko.Transport(sock, *args, **kwargs)
        options = transport.get_security_options()
        if key_type and key_type in options.key_types:
            options.key_types = [key_type] + [k for k in options.key_types if k != key_type]
        return transport

    return transport_factory


def build_ssh_client():
    client = paramiko.SSHClient()
    if sftp_host_key_fingerprint:
        client.set_missing_host_key_policy(FingerprintHostKeyPolicy(sftp_host_key_fingerprint))
    else:
        print('WARNING: SFTP_SSHHOSTKEYFINGERPRINT is not set, falling back to ~/.ssh/known_hosts.')
        client.load_system_host_keys()
        client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    return client


def open_sftp_connection():
    """Open an SFTP session on SFTP_HOST as SFTP_LOGIN.

    First choice: SSH public-key authentication, with the server pinned to
    SFTP_SSHHOSTKEYFINGERPRINT. Second choice: the historical SFTP_PASSWORD.
    A host key mismatch is never retried - it aborts the connection.
    """
    key_files = collect_private_key_files()
    attempts = [(
        'SSH key',
        {
            'key_filename': key_files or None,
            'passphrase': resolve_key_passphrase(key_files),
            'allow_agent': True,
            'look_for_keys': True,
        },
    )]
    if sftp_password:
        attempts.append((
            'password',
            {'password': sftp_password, 'allow_agent': False, 'look_for_keys': False},
        ))

    common_kwargs = {
        'hostname': sftp_host,
        'port': sftp_port,
        'username': sftp_login,
        'timeout': 30,
        'transport_factory': make_transport_factory(),
    }

    last_error = None
    for label, auth_kwargs in attempts:
        client = build_ssh_client()
        try:
            client.connect(**common_kwargs, **auth_kwargs)
        except (HostKeyVerificationError, paramiko.BadHostKeyException):
            client.close()
            raise
        except (paramiko.AuthenticationException, paramiko.SSHException) as exc:
            client.close()
            print(f'SFTP {label} authentication failed: {type(exc).__name__}: {exc}')
            last_error = exc
            continue

        print(f'SFTP connected using {label} authentication')
        return client, client.open_sftp()

    raise last_error or paramiko.AuthenticationException('No SFTP authentication method succeeded')


def download_new_csv_files(remote_folder, local_folder):
    remote_path = posixpath.join(sftp_base_folder, remote_folder)
    local_path = Path('./data') / local_folder
    local_path.mkdir(parents=True, exist_ok=True)

    downloaded_count = 0
    skipped_count = 0

    ssh_client, sftp = open_sftp_connection()

    try:
        for entry in sftp.listdir_attr(remote_path):
            if entry.filename in ('.', '..'):
                continue

            remote_file = posixpath.join(remote_path, entry.filename)
            local_file = local_path / entry.filename

            try:
                if (entry.st_mode & 0o170000) == 0o040000:
                    continue
            except Exception:
                pass

            if local_file.exists():
                skipped_count += 1
                continue

            sftp.get(remote_file, str(local_file))
            downloaded_count += 1
    finally:
        sftp.close()
        ssh_client.close()

    print(f'SFTP folder: {remote_path}')
    print(f'Local folder: {local_path.resolve()}')
    print(f'Downloaded {downloaded_count} new file(s), skipped {skipped_count} existing file(s)')


def get_latest_csv(local_folder, local_file_basename):
    base_candidates = [Path('./data') / local_folder, Path('./data')]

    csv_candidates = []
    for base in base_candidates:
        if base.exists():
            csv_candidates.extend(base.glob(local_file_basename + '*.csv'))

    if not csv_candidates:
        raise FileNotFoundError(
            'No file found matching ' + local_file_basename + '*.csv in ./data or ./data/<local_folder>'
        )

    # The export date lives in the filename; mtime is only the SFTP download
    # time, identical across a whole mirror pass, so it cannot order the exports.
    latest_csv = max(csv_candidates, key=lambda p: (p.name, str(p)))
    print(f'Using latest CSV: {latest_csv}')
    return latest_csv


def set_store_value(store_key, value):
    ip = get_ipython()
    if ip is None:
        return

    ip.user_ns[store_key] = value
    ip.run_line_magic('store', store_key)


def get_store_value(store_key, default=0):
    """Read back the cursor persisted by set_store_value.

    This is the resume half of the %store pair: without it the loop always
    restarts at the top of the CSV. Reading ip.db directly rather than
    running '%store -r' keeps an unknown key from raising on a first run.
    """
    ip = get_ipython()
    if ip is None:
        return default

    try:
        return ip.db['autorestore/' + store_key]
    except KeyError:
        return default


def apply_http_client_timeout(driver, timeout=HTTP_CLIENT_TIMEOUT):
    """Shorten the read timeout of the HTTP connection to chromedriver.

    ChromiumRemoteConnection hard-codes 120 s and webdriver.Chrome takes no
    client_config, so the value is set on the live connection: selenium passes
    client_config.timeout on every request, so it applies from the next command
    on. Without it a crashed Chrome only surfaces two minutes later, as a raw
    urllib3 ReadTimeoutError.
    """
    try:
        driver.command_executor.client_config.timeout = timeout
    except AttributeError as exc:
        print(f'Could not shorten the chromedriver HTTP timeout ({exc}).')


def build_chrome_options():
    """Chrome flags that slow the memory growth behind the out-of-memory crashes.

    The back/forward cache keeps each previous page alive inside the renderer the
    whole run shares, which is memory the loop never gets back: 12 measured
    navigations cost 16 MB per page with it on, 9 MB with it off.
    """
    options = webdriver.ChromeOptions()
    options.add_argument('--disable-features=BackForwardCache')
    return options


def purge_browser_memory(driver):
    """Ask Chrome to drop what it can of the renderer memory, between records.

    One CDP call, and measurable: with the back/forward cache already off it took
    the growth from 9 MB to 4 MB per navigation. Failing is harmless, so a driver
    that does not support it is simply left alone.
    """
    try:
        driver.execute_cdp_cmd('Memory.forciblyPurgeJavaScriptMemory', {})
    except WebDriverException:
        pass


def init_driver():
    driver = webdriver.Chrome(
        service=ChromeService(ChromeDriverManager().install()),
        options=build_chrome_options()
    )
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    driver.set_script_timeout(SCRIPT_TIMEOUT)
    apply_http_client_timeout(driver)
    return driver


def get_chrome_error_code(driver):
    """Return Chrome's error code when the page shown is Chrome's own error page.

    This is the failure that goes unnoticed: after a renderer runs out of memory
    Chrome replaces the page with its 'Aw, Snap!' document, served by the browser
    itself. driver.get() returns, every command keeps working, nothing raises,
    and a loop that does not look for it skips record after record convinced it
    saw a page. driver.current_url still reports the URL that was asked for, so
    document.URL is what gives it away (chrome-error://chromewebdata/), together
    with the neterror body class and the error code the page displays.
    """
    try:
        state = driver.execute_script(
            'return [document.URL, document.body ? document.body.className : ""];'
        )
    except WebDriverException:
        return None

    if not state:
        return None

    document_url = state[0] or ''
    body_class = state[1] or ''
    if not document_url.startswith('chrome-error://') and 'neterror' not in body_class:
        return None

    codes = driver.find_elements(By.CSS_SELECTOR, 'div.error-code')
    return codes[0].text.strip() if codes else 'unknown'


def open_page(driver, url):
    """Navigate to url, and refuse to carry on over one of Chrome's error pages."""
    timed_out = False
    try:
        driver.get(url)
    except TimeoutException:
        print(f'Page load timed out: {url}')
        timed_out = True
        try:
            driver.execute_script('window.stop();')
        except WebDriverException:
            pass

    error_code = get_chrome_error_code(driver)
    if error_code is not None:
        raise BrowserSessionLost(f'Chrome error page ({error_code})')

    return not timed_out


def open_edit_page(driver, entity_path, lngid):
    return open_page(
        driver,
        f'https://www.themoviedb.org/{entity_path}/{lngid}/edit?active_nav_item=external_ids'
    )


def dismiss_cookie_banner(driver, timeout=COOKIE_BANNER_TIMEOUT):
    """Decline TMDB's cookie banner, which otherwise covers the Save button.

    Returns False when no banner shows up, which is the normal case on a browser
    that already declined: the caller has nothing to do about it.
    """
    try:
        button = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((By.ID, COOKIE_DECLINE_ID))
        )
    except TimeoutException:
        return False

    try:
        button.click()
    except (ElementClickInterceptedException, ElementNotInteractableException):
        driver.execute_script('arguments[0].click();', button)

    try:
        WebDriverWait(driver, timeout).until(
            EC.invisibility_of_element_located((By.ID, COOKIE_BANNER_ID))
        )
    except TimeoutException:
        print('Cookie banner still visible after clicking Decline all.')
        return False

    print('Cookie banner declined.')
    return True


def click_save_button(driver):
    """Click Save, clearing the cookie banner out of the way if it intercepts it."""
    try:
        driver.find_element(By.XPATH, SAVE_BUTTON_XPATH).click()
        return
    except ElementClickInterceptedException:
        print('Save click intercepted: dismissing the cookie banner.')

    dismiss_cookie_banner(driver)
    save_button = driver.find_element(By.XPATH, SAVE_BUTTON_XPATH)
    try:
        save_button.click()
    except ElementClickInterceptedException:
        # The button is there and enabled, something is just sitting on top of
        # it: a scripted click reaches it whatever that something is.
        driver.execute_script('arguments[0].click();', save_button)


def login_tmdb(driver):
    open_page(driver, 'https://www.themoviedb.org/login')
    dismiss_cookie_banner(driver)
    time.sleep(10)

    try:
        username_field = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.NAME, 'username'))
        )
    except TimeoutException:
        # Raising here would abort a run that a restart was trying to save.
        print('TMDB login form not found: check the Chrome window.')
        return False

    username_field.clear()
    username_field.send_keys(tmdb_login)

    password_field = driver.find_element(By.NAME, 'password')
    password_field.clear()
    password_field.send_keys(tmdb_password)
    password_field.send_keys('\n')

    return wait_until_logged_in(driver)


def wait_until_logged_in(driver, timeout=LOGIN_CONFIRM_TIMEOUT):
    """Wait for TMDB to leave the login page.

    A successful login redirects away from /login, a CAPTCHA or a refused login
    stays on it. The URL is the signal here rather than is_logged_out(), because
    the anonymous home page carries a login form of its own in the header.
    """
    deadline = time.time() + timeout
    warn_at = time.time() + 15
    warned = False

    while time.time() < deadline:
        if '/login' not in driver.current_url:
            print('TMDB login confirmed.')
            return True

        if not warned and time.time() >= warn_at:
            print('Still on the TMDB login page: solve the CAPTCHA in the Chrome window if one is shown.')
            warned = True

        time.sleep(5)

    print(f'TMDB login still not confirmed after {timeout} s.')
    return False


def is_browser_gone(exc):
    """True when the exception means Chrome or chromedriver stopped answering.

    A live browser raises selenium errors (a missing element, a slow page); a
    crashed one answers with an invalid session, or stops answering at all, and
    that last case surfaces as a raw urllib3 read timeout, which is not a
    WebDriverException.
    """
    if isinstance(exc, (InvalidSessionIdException, NoSuchWindowException)):
        return True

    if isinstance(exc, (urllib3.exceptions.HTTPError, ConnectionError, TimeoutError)):
        return True

    if isinstance(exc, WebDriverException):
        message = (exc.msg or str(exc)).lower()
        return any(marker in message for marker in BROWSER_GONE_MESSAGES)

    return False


def shutdown_browser(driver, timeout=BROWSER_QUIT_TIMEOUT):
    """Close Chrome without letting a hung driver block the run.

    driver.quit() talks to a chromedriver that may never answer, so it runs in a
    daemon thread and the process is killed when it does not return in time.
    """
    def quit_quietly():
        try:
            driver.quit()
        except Exception:
            pass

    thread = threading.Thread(target=quit_quietly, daemon=True)
    thread.start()
    thread.join(timeout)
    if not thread.is_alive():
        return

    print('Chrome did not close in time: killing chromedriver. A leftover Chrome window may need closing by hand.')
    process = getattr(getattr(driver, 'service', None), 'process', None)
    if process is not None and process.poll() is None:
        try:
            process.kill()
        except OSError as exc:
            print(f'chromedriver could not be killed ({exc}).')


def restart_browser(driver):
    """Replace the current browser with a fresh, logged-in one.

    Returns the new driver: the old one points at a session that no longer
    exists, so every caller must rebind.
    """
    shutdown_browser(driver)
    time.sleep(3)
    new_driver = init_driver()
    try:
        logged_in = login_tmdb(new_driver)
    except BrowserSessionLost as exc:
        # The caller counts the attempts and stops the run; raising from here
        # would abort with the login error instead of that clearer message.
        logged_in = False
        print(f'Login page could not be reached after the restart ({exc}).')

    if not logged_in:
        print('TMDB login not confirmed after the restart: check the Chrome window.')

    return new_driver


def get_log_file_path():
    log_dir = Path('./data')
    log_dir.mkdir(parents=True, exist_ok=True)
    return log_dir / 'selenium-tmdb-wikidata_id.log'


def append_processed_log(content_type, record_id, wikidata_id):
    log_file_path = get_log_file_path()
    with log_file_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f'{content_type};{record_id};{wikidata_id}\n')


def is_page_not_found(driver):
    elements = driver.find_elements(By.XPATH, "//h2[text()=\"Oops! We can't find the page you're looking for\"]")
    return len(elements) > 0


def is_logged_out(driver):
    return len(driver.find_elements(By.NAME, 'password')) > 0


def is_wikidata_id_locked(driver):
    # Every field carries a #wikidata_id_status padlock control: the discriminating
    # class is 'locked', never 'locked_status' alone.
    css_selectors = (
        'span#wikidata_id_status.glyphicons_v2.locked.locked_status',
        'span#wikidata_id_status.locked',
    )
    for css_selector in css_selectors:
        if len(driver.find_elements(By.CSS_SELECTOR, css_selector)) > 0:
            return True
    return False


def find_visible_element(driver, by, value):
    for element in driver.find_elements(by, value):
        try:
            if element.is_displayed():
                return element
        except StaleElementReferenceException:
            continue
    return None


def activate_external_ids_panel(driver):
    """The edit page keeps every section in the DOM and only shows the active one.

    When the active_nav_item URL parameter does not take effect, click the
    'External IDs' nav link so the wikidata_id input becomes interactable.
    """
    link = find_visible_element(driver, By.CSS_SELECTOR, 'a[href*="active_nav_item=external_ids"]')
    if link is None:
        return False

    driver.execute_script('arguments[0].click();', link)
    return True


def get_wikidata_id_field(driver, lngid, timeout_message):
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.NAME, 'wikidata_id'))
        )
    except TimeoutException:
        print(timeout_message.format(lngid=lngid))
        return None

    # The panel is rendered asynchronously and several sections may each hold a
    # wikidata_id input: keep the visible one.
    try:
        field = WebDriverWait(driver, 5).until(
            lambda d: find_visible_element(d, By.NAME, 'wikidata_id')
        )
    except TimeoutException:
        field = None

    if field is None and activate_external_ids_panel(driver):
        print("Panel 'External IDs' activated manually.")
        try:
            field = WebDriverWait(driver, 10).until(
                lambda d: find_visible_element(d, By.NAME, 'wikidata_id')
            )
        except TimeoutException:
            field = None

    if field is None:
        # Fall back to the first match so the caller can report its actual state.
        field = driver.find_element(By.NAME, 'wikidata_id')

    return field


def describe_field_state(field):
    try:
        return (
            f"displayed={field.is_displayed()} enabled={field.is_enabled()} "
            f"readonly={field.get_attribute('readonly')}"
        )
    except StaleElementReferenceException:
        return 'stale element'


def is_field_writable(field):
    try:
        return field.is_displayed() and field.is_enabled() and field.get_attribute('readonly') is None
    except StaleElementReferenceException:
        return False


def write_wikidata_id(field, strwikidataid=None):
    """Clear the field, then type strwikidataid when provided.

    Returns False (instead of raising) when TMDB refuses the edit, so the batch
    loop can skip the record and carry on.
    """
    try:
        field.clear()
        if strwikidataid is not None:
            field.send_keys(strwikidataid)
        return True
    except (
        InvalidElementStateException,
        ElementNotInteractableException,
        StaleElementReferenceException,
    ) as exc:
        print(f'wikidata_id field could not be edited ({type(exc).__name__}).')
        return False


def confirm_wikidata_id(driver, expected_value):
    """Re-read the field after Save so only real writes are reported and logged."""
    def has_expected_value(d):
        field = find_visible_element(d, By.NAME, 'wikidata_id')
        if field is None:
            return False
        try:
            return field.get_attribute('value') == expected_value
        except StaleElementReferenceException:
            return False

    try:
        return WebDriverWait(driver, 10).until(has_expected_value)
    except TimeoutException:
        return False


def debug_wikidata_page(driver, entity_path, lngid):
    """Dump the state of one edit page. Read-only: nothing is written to TMDB."""
    open_edit_page(driver, entity_path, lngid)
    time.sleep(3)

    print(f'url          : {driver.current_url}')
    print(f'title        : {driver.title}')
    print(f'logged out   : {is_logged_out(driver)}')
    print(f'page 404     : {is_page_not_found(driver)}')
    print(f'locked       : {is_wikidata_id_locked(driver)}')
    print(f'save buttons : {len(driver.find_elements(By.XPATH, SAVE_BUTTON_XPATH))}')

    fields = driver.find_elements(By.NAME, 'wikidata_id')
    print(f'wikidata_id inputs: {len(fields)}')
    for position, field in enumerate(fields):
        print(f'  [{position}] {describe_field_state(field)} value={field.get_attribute("value")!r}')
        print(f'       {field.get_attribute("outerHTML")[:400]}')

    for status in driver.find_elements(By.CSS_SELECTOR, '#wikidata_id_status'):
        print(f'  status: {status.get_attribute("outerHTML")[:300]}')


def set_wikidata_id(driver, entity_path, lngid, strwikidataid, timeout_message):
    if not open_edit_page(driver, entity_path, lngid):
        return False

    intpagefound = True
    try:
        if is_page_not_found(driver):
            print("Element 'Page 404' exists on the page.")
            intpagefound = False
        else:
            print("Element 'Page 404' does not exist on the page.")
    except NoSuchElementException:
        print("Element 'Page 404' does not exist on the page.")

    if intpagefound:
        xpath = "//button[span[contains(@class, 'glyphicons_v2') and contains(@class, 'plus') and contains(@class, 'svg')] and contains(., 'Create Translation')]"
        try:
            button = driver.find_element(By.XPATH, xpath)
            button.click()
            print("Button 'Create translation' clicked successfully.")
            time.sleep(2)
            open_edit_page(driver, entity_path, lngid)
        except Exception:
            print("Button 'Create translation' not found, translation already exists.")

        wikidata_id_field = get_wikidata_id_field(driver, lngid, timeout_message)
        if wikidata_id_field is None:
            if is_logged_out(driver):
                raise BrowserSessionLost('TMDB redirected to the login page')
            return False

        if is_wikidata_id_locked(driver):
            print("Element 'Locked' exists on the page.")
            return False

        if not is_field_writable(wikidata_id_field):
            print(
                f'wikidata_id field is not editable for {entity_path} {lngid} '
                f'({describe_field_state(wikidata_id_field)}). Skipping.'
            )
            return False

        if not write_wikidata_id(wikidata_id_field, strwikidataid):
            print(f'Skipping {entity_path} {lngid}.')
            return False

        click_save_button(driver)

        if not confirm_wikidata_id(driver, strwikidataid):
            print(f'Save not confirmed for {entity_path} {lngid}: {strwikidataid} was not applied.')
            return False

        print(f'{entity_path} {lngid} updated with {strwikidataid}.')
        return True

    return False


def clear_wikidata_id(driver, entity_path, lngid):
    if not open_edit_page(driver, entity_path, lngid):
        return False

    if is_page_not_found(driver):
        print("Element 'Page 404' exists on the page. Skipping clear.")
        return False

    wikidata_id_field = get_wikidata_id_field(
        driver, lngid, f'wikidata_id field not found for {entity_path} {{lngid}}. Skipping clear.'
    )
    if wikidata_id_field is None:
        if is_logged_out(driver):
            raise BrowserSessionLost('TMDB redirected to the login page')
        return False

    if is_wikidata_id_locked(driver):
        print("Element 'Locked' exists on the page. Skipping clear.")
        return False

    if not is_field_writable(wikidata_id_field):
        print(
            f'wikidata_id field is not editable for {entity_path} {lngid} '
            f'({describe_field_state(wikidata_id_field)}). Skipping clear.'
        )
        return False

    if not write_wikidata_id(wikidata_id_field):
        print(f'Skipping clear for {entity_path} {lngid}.')
        return False

    click_save_button(driver)

    if not confirm_wikidata_id(driver, ''):
        print(f'Clear not confirmed for {entity_path} {lngid}.')
        return False

    return True


def process_row(driver, dataset, lngid, strwikidataid, strtmdbidtoerase):
    """Apply one CSV row to TMDB, assuming a live browser.

    Raises instead of returning when the browser or the session is gone, so the
    caller restarts Chrome and tries the row again rather than storing a cursor
    past a record that was never written.
    """
    if pd.isna(strtmdbidtoerase):
        print('strtmdbidtoerase is NaN')
    else:
        print('strtmdbidtoerase is not NaN')
        lngtmdbidtoerase = int(strtmdbidtoerase)
        clear_wikidata_id(driver, dataset['entity_path'], lngtmdbidtoerase)

    return set_wikidata_id(
        driver,
        dataset['entity_path'],
        lngid,
        strwikidataid,
        dataset['timeout_message']
    )


def process_row_with_recovery(driver, dataset, lngid, strwikidataid, strtmdbidtoerase):
    """Run process_row, restarting Chrome when it dies under the row.

    Returns (driver, was_updated). After a restart the driver is a new one, so
    the caller must keep what comes back: the old session no longer exists.
    """
    for attempt in range(1, BROWSER_RESTART_ATTEMPTS + 1):
        try:
            return driver, process_row(driver, dataset, lngid, strwikidataid, strtmdbidtoerase)
        except BrowserSessionLost as exc:
            reason = str(exc)
        except Exception as exc:
            if not is_browser_gone(exc):
                raise
            reason = f'{type(exc).__name__}: {exc}'.strip()

        entity_path = dataset['entity_path']
        print(
            f'Browser lost on {entity_path} {lngid} ({reason}). '
            f'Attempt {attempt}/{BROWSER_RESTART_ATTEMPTS}.'
        )
        if attempt == BROWSER_RESTART_ATTEMPTS:
            break

        driver = restart_browser(driver)

    raise RuntimeError(
        f"Chrome could not be kept alive for {dataset['entity_path']} {lngid} after "
        f"{BROWSER_RESTART_ATTEMPTS} attempts. The {dataset['store_key']} cursor is stored: "
        'fix the browser, then run the cell again to resume where it stopped.'
    )


def process_dataset(driver, dataset):
    """Walk one CSV and apply every row, returning the driver still in use.

    Chrome is restarted in place when it crashes, so the caller must rebind:
    driver = process_dataset(driver, dataset).
    """
    latest_csv = get_latest_csv(dataset['local_folder'], dataset['local_file_basename'])
    data = pd.read_csv(str(latest_csv), sep=';', quotechar='"')
    print(data.shape)

    store_key = dataset['store_key']
    start_value = get_store_value(store_key, 0)
    print(f'{store_key} = {start_value}')
    processed_count = 0
    consecutive_skips = 0
    skip_restarts = 0

    if dataset.get('name') == 'persons' and not dataset.get('intgoingdown', True):
        rows = data.iloc[::-1].iterrows()
        comparator = lambda current_id, last_id: current_id < last_id
    else:
        rows = data.iterrows()
        comparator = lambda current_id, last_id: current_id > last_id

    for index, row in rows:
        print(f'Index: {index} ; processed: {processed_count}')

        if row[dataset['id_column']]:
            lngid = row[dataset['id_column']]
            if row['ID_WIKIDATA']:
                strwikidataid = row['ID_WIKIDATA']
                strtmdbidtoerase = row[dataset['erase_column']]
                if comparator(lngid, start_value):
                    driver, was_updated = process_row_with_recovery(
                        driver, dataset, lngid, strwikidataid, strtmdbidtoerase
                    )
                    if was_updated:
                        append_processed_log(dataset['name'], lngid, strwikidataid)
                        consecutive_skips = 0
                        skip_restarts = 0
                    else:
                        consecutive_skips += 1

                    processed_count += 1
                    start_value = lngid
                    set_store_value(store_key, start_value)
                    purge_browser_memory(driver)

                    if consecutive_skips >= MAX_CONSECUTIVE_SKIPS:
                        if skip_restarts:
                            raise RuntimeError(
                                f'{consecutive_skips} records in a row wrote nothing, and a fresh '
                                f'browser did not change that. The {store_key} cursor is stored: '
                                'check TMDB in the window before resuming.'
                            )
                        print(f'{consecutive_skips} records in a row wrote nothing: restarting Chrome.')
                        skip_restarts += 1
                        consecutive_skips = 0
                        driver = restart_browser(driver)
                    elif BROWSER_RECYCLE_EVERY and processed_count % BROWSER_RECYCLE_EVERY == 0:
                        print(f'{processed_count} records processed: recycling Chrome.')
                        driver = restart_browser(driver)

                    time.sleep(2)
            else:
                print(dataset['missing_wikidata_message'])
        else:
            print(dataset['missing_entity_message'])

    return driver


In [6]:
if False:
    for dataset in DATASETS:
        download_new_csv_files(dataset['remote_folder'], dataset['local_folder'])

driver = init_driver()
login_tmdb(driver)


Cookie banner declined.
TMDB login confirmed.


True

In [7]:
# Diagnostic (read-only, no write to TMDB): inspect one edit page to understand
# why the wikidata_id field is refused. Run it after the login cell.
debug_wikidata_page(driver, 'movie', 320496)


url          : https://www.themoviedb.org/movie/320496-thaaliya-bhagya/edit?active_nav_item=external_ids
title        : Edit Thaaliya Bhagya — The Movie Database (TMDB)
logged out   : False
page 404     : False
locked       : False
save buttons : 1
wikidata_id inputs: 1
  [0] displayed=True enabled=True readonly=None value='Q13159526'
       <input dir="auto" id="wikidata_id" class="k-input k-input-solid k-input-md k-rounded-md k-input-inner" type="text" name="wikidata_id" value="Q13159526" autocomplete="off" data-role="textbox" aria-disabled="false" inputmode="text" style="width: 100%;">
  status: <span id="wikidata_id_status" class="glyphicons unlocked locked_status mr-0!"></span>


In [ ]:
# process_dataset returns the driver in use: Chrome is restarted in place when
# it crashes, and the old session dies with it.
driver = process_dataset(driver, DATASETS[0])
driver = process_dataset(driver, DATASETS[1])
driver = process_dataset(driver, DATASETS[2])


Using latest CSV: data\wikidata-id-movie-fix\wikidata-id-movie-fix-20260902.csv
(20255, 8)
lngmovieidstart = 1760337
Index: 0 ; processed: 0
Index: 1 ; processed: 0
Index: 2 ; processed: 0
Index: 3 ; processed: 0
Index: 4 ; processed: 0
Index: 5 ; processed: 0
Index: 6 ; processed: 0
Index: 7 ; processed: 0
Index: 8 ; processed: 0
Index: 9 ; processed: 0
Index: 10 ; processed: 0
Index: 11 ; processed: 0
Index: 12 ; processed: 0
Index: 13 ; processed: 0
Index: 14 ; processed: 0
Index: 15 ; processed: 0
Index: 16 ; processed: 0
Index: 17 ; processed: 0
Index: 18 ; processed: 0
Index: 19 ; processed: 0
Index: 20 ; processed: 0
Index: 21 ; processed: 0
Index: 22 ; processed: 0
Index: 23 ; processed: 0
Index: 24 ; processed: 0
Index: 25 ; processed: 0
Index: 26 ; processed: 0
Index: 27 ; processed: 0
Index: 28 ; processed: 0
Index: 29 ; processed: 0
Index: 30 ; processed: 0
Index: 31 ; processed: 0
Index: 32 ; processed: 0
Index: 33 ; processed: 0
Index: 34 ; processed: 0
Index: 35 ; proces

In [ ]:
# Loop is finished so we display the home page
open_page(driver, 'https://www.themoviedb.org/')